# Pendulum

**AlphaLabs** is a collection of computer simulations built to enhance the physics learning experience for university students. It offers pre-laboratory simulations that visually demonstrate physics concepts and prepare students for hands-on labs.

Repository: https://github.com/commanderxa/alphalabs<br>
Website: https://commanderxa.github.io/alphalabs/

Visit the website for more info.

### **Introduction**



### **What is it?**



### **Example**



# Simulation

## Setup

Installing packages (for Google Colab). If this notebook is opened in Google Colab then some packages must be installed to run the code!\
Then importing the scene and plot settings.

In [ ]:
#@title Run to install MuJoCo and `dm_control` for Google Colab

IS_COLAB = 'google.colab' in str(get_ipython())
if IS_COLAB:
    # download the necessary files from the repository
    !wget -nv https://raw.githubusercontent.com/commanderxa/alphalabs/main/mechanics/setup.py
    !wget -nv https://raw.githubusercontent.com/commanderxa/alphalabs/main/mechanics/scene.py
    !wget -nv https://raw.githubusercontent.com/commanderxa/alphalabs/main/mechanics/utils.py
    from setup import install_packages_colab
    install_packages_colab()
else:
    import os, sys
    module_path = os.path.abspath(os.path.join(".."))
    if module_path not in sys.path:
        sys.path.append(module_path)

## Import

Import all required packages to preform simulations. Packages include simulation engine, plotting libraries and other ones necessary for computations.

In [ ]:
%env MUJOCO_GL=egl

# simulation
from dm_control import mjcf
# for video recording
import mediapy
# computations
import numpy as np
# plot charts
import seaborn as sns
import matplotlib.pyplot as plt
# helper files
from scene import Scene
import utils

## Initial Conditions

In this block constants are defined. They impact the environment, rendering and objects directly.

In [ ]:
# global
IS_IDEAL = True  # specifies if the world is ideal (e.g. no frictions, etc.)
VISCOSITY = 0.00002  # air resistance
DENSITY = 1.2  # air density

# simulation constants
LENGTH = 0.25
ANGLE: float = 10  # initial angle of release (in degrees)
MASS: float = 0.1  # kg
GAP: float = 0.05  # gap between pendulums (in meters)

# engine
INTEGRATOR = "RK4"
INTEGRATOR_TIMESTEP = 0.001

# rendering
WIDTH = 1280
HEIGHT = 720
DPI = 600
DURATION = 30  # (seconds)
FRAMERATE = 60  # (Hz)

LAB_NAME = "pendulum_swing"
LAB_PATH = f"{LAB_NAME}/{'ideal' if IS_IDEAL else 'real'}"

## Model

### Objects of Interest

This class defines the object of our interest, a `pendulum`. Here we write what is this object (box), what can it do (swing).

In [ ]:
class Pendulum(object):
    """Pendulum MJCF model"""

    def __init__(self, mass: float, length: float, rgba: list) -> None:
        self.model = mjcf.RootElement(model="pendulum")

        self.damping = 0 if IS_IDEAL else 0.001642

        self.pendulum = self.model.worldbody.add("body", name="pendulum", pos=[0, 0, 0], euler=[0, ANGLE, 0])
        self.anchor = self.pendulum.add("site", name="anchor", pos=[0, 0, 0], size=[0.01])
        self.swing = self.pendulum.add("joint", name="swing", axis=[0, 1, 0], damping=self.damping)

        self.bob = self.pendulum.add(
            "body", name="bob", pos=[0, 0, -length], euler=[0, 0, 0]
        )
        self.sphere = self.bob.add(
            "geom", name="sphere", size=[0.025], pos=[0, 0, 0], rgba=rgba, mass=mass
        )
        self.hook = self.bob.add("site", name="hook", pos=[0, 0, 0], size=[0.001])

        self.tendon = self.model.tendon.add("spatial")
        self.tendon.width = 0.0025
        self.tendon.add("site", site="anchor")
        self.tendon.add("site", site="hook")

### World Model

Collecting everything into one general model

In [ ]:
class Model(object):

    def __init__(self, mass: float, length: float) -> None:
        self.model = mjcf.RootElement(model="model")

        # set render info
        self.model.visual.__getattr__("global").offheight = HEIGHT
        self.model.visual.__getattr__("global").offwidth = WIDTH

        # set the simulation constants
        self.model.option.integrator = INTEGRATOR
        self.model.option.timestep = INTEGRATOR_TIMESTEP
        if IS_IDEAL:
            self.model.option.viscosity = 0
            self.model.option.density = 0
        else:
            self.model.option.viscosity = VISCOSITY
            self.model.option.density = DENSITY

        # create the scene (ground)
        self.scene = Scene(length=10)
        self.scene_site = self.model.worldbody.add("site", pos=[0, 0, -length*1.5])
        self.scene_site.attach(self.scene.model)

        # create a group of pendulums
        pendulum = Pendulum(mass, length, rgba=[1, 0, 0, 1])
        pendulum_wave_site = self.model.worldbody.add("site", pos=[0, 0, 0])
        pendulum_wave_site.attach(pendulum.model)

        # define camera position
        cam_x = 0
        cam_y = 1.6 * length  # Pull back farther if needed
        cam_z = -0.55 * length
        # define camera
        self.camera = self.model.worldbody.add(
            "camera",
            name="front",
            pos=[cam_x, cam_y, cam_z],
            euler=[-90, 0, 180],
            mode="fixed",
        )

## Simulation

Initializing the `physics` of the simulation

In [ ]:
model = Model(mass=MASS, length=LENGTH).model
physics = mjcf.Physics.from_mjcf_model(model)

First of all, the environment must be verified by rendering a picture

In [ ]:
mediapy.show_image(physics.render(height=320, width=640, camera_id=0))
mediapy.write_image(utils.prefix_path("preview.png", LAB_PATH), physics.render(height=HEIGHT, width=WIDTH, camera_id=0))

### Simulation Loop

In [ ]:
physics.reset()

# intiialize empty arrays to record data
frames = []
timevals = []
velocity = []
position = []

while physics.time() < DURATION:

    # record data
    timevals.append(physics.time())
    velocity.append(physics.named.data.qvel.copy())
    position.append(physics.named.data.geom_xpos[1, 0].copy())

    # render a picture
    if len(frames) < physics.time() * FRAMERATE:
        pixels = physics.render(width=WIDTH, height=HEIGHT, camera_id=0)
        frames.append(pixels)
    
    # update the simulation
    physics.step()

Show and save video of the simulation

In [ ]:
mediapy.show_video(frames, fps=FRAMERATE)
mediapy.write_video(utils.prefix_path("simulation.mp4", LAB_PATH), images=frames, fps=FRAMERATE)

## Data Analysis

Convert data into numpy arrays

In [ ]:
timevals = np.array(timevals)
velocity = np.array(velocity).squeeze(-1)
position = np.array(position)

Plot the chart with the collected data

In [ ]:
cm = 1 / 2.54
figsize = (8.5 * cm, 3.75 * cm)
fig, ax = plt.subplots(ncols=2, figsize=figsize, dpi=DPI, sharey=True)
fig.subplots_adjust(wspace=0.2 * cm)

ax[0].plot(velocity, timevals, linewidth=0.5, color="tab:blue")
ax[0].set_xlabel('Angular Velocity, [m/s]', fontsize=7, labelpad=2)
ax[0].set_ylabel('Time, [s]', fontsize=7, labelpad=2)
ax[0].invert_yaxis()
ax[0].tick_params(labelsize=6, which="both", direction="in", top=True, right=True, length=2)

ax[1].plot(position, timevals, linewidth=0.5, color="tab:blue")
ax[1].set_xlabel('Position, [m]', fontsize=7, labelpad=2)
ax[1].tick_params(labelsize=6, which="both", direction="in", top=True, right=True, length=2)

fig.savefig(utils.prefix_path("chart.png", LAB_PATH), bbox_inches="tight")